In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd

In [ ]:
# where WellClass and Ga[ codes are located
sys.path.append('../')

In [ ]:
from src.WellClass.libs.models.well_model import WellModel
from src.WellClass.libs.well_class   import WellProcessed

from src.WellClass.libs.well_class   import Well
from src.WellClass.libs.plotting.plot_sketch import plot_sketch

In [ ]:
frigg_json = '../test_data/examples/frigg/frigg.json'

json_data = json.load(open(frigg_json))


my_model = WellModel(**json_data)

In [ ]:
json_data['spec'].keys()

In [ ]:
my_well = WellProcessed.from_json(frigg_json)

In [ ]:
my_well

In [ ]:
my_well.plot_sketch()

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any, Union
import numpy as np
from scipy import constants as const

def shmin_data_interpolator(shmin_data:List[List[float]], depth_array:np.ndarray, ground_elevation:float, ground_pressure_bar:float, surface_pressure_bar:float) -> np.ndarray:

    shmin_data = np.array(shmin_data)
    depth_values, shmin_values = shmin_data.T

    # Remove duplicates while keeping the first occurrence
    unique_indices = np.unique(depth_values, return_index=True)[1]
    depth_values = depth_values[np.sort(unique_indices)]
    shmin_values = shmin_values[np.sort(unique_indices)]


    if min(depth_values) > ground_elevation:
        warnings.warn(
            f"No Shmin data between seafloor depth ({ground_elevation}) and minimum provided depth ({min(depth_values)}). "
            "Extrapolating using hydrostatic pressure at seafloor."
        )

    else:
        filtered_depth_values = depth_values[depth_values >= ground_elevation]
        filtered_shmin_values = shmin_values[depth_values >= ground_elevation]

        depth_values = filtered_depth_values
        shmin_values = filtered_shmin_values

    # Insert the seafloor depth and pressure at mudline into the arrays
    depth_values = np.insert(depth_values, 0, ground_elevation)
    shmin_values = np.insert(shmin_values, 0, ground_pressure_bar)


    # Insert values af MSL
    depth_values = np.insert(depth_values, 0, 0)
    shmin_values = np.insert(shmin_values, 0, surface_pressure_bar)


    shmin_interpolator = interp1d(depth_values, shmin_values, bounds_error=False, fill_value="extrapolate")

    shmin_curve = shmin_interpolator(depth_array)
    return shmin_curve



@dataclass
class PressureTable:
    name: str
    depth: np.ndarray  # depth array (m)

    # Input parameters
    ground_elevation: float  # ground elevation (m)
    ground_temperature: float  # ground temperature (°C)
    geothermal_gradient: float  # geothermal gradient (°C/km)
    rho_brine : float = 1030  # brine density (kg/m³)

    # Minimum horizontal stress parameters
    shmin_gradient: float = 0.1695
    shmin_data: List[List[float]] = field(default=None)  # depth values for shmin data

    # Computed arrays
    temperature: np.ndarray = field(init=False) # temperature array (°C)
    hydrostatic_pressure: np.ndarray = field(init=False)   # hydrostatic pressure (MPa or bar)
    min_horizontal_stress: np.ndarray  = field(init=False) # minimum horizontal stress (MPa or bar)
    # fluid_pressure: np.ndarray  # fluid pressure (MPa or bar)
    # brine_pressure: np.ndarray  # brine pressure (MPa or bar)
    # min_horizontal_stress: np.ndarray  # minimum horizontal stress (MPa or bar)

    def __post_init__(self):
        # Compute temperature array based on depth and geothermal gradient
        self.temperature = self.compute_temperature()
        self.hydrostatic_pressure = self.compute_hydrostatic_pressure()
        self.min_horizontal_stress = self.compute_shmin()
    #     lengths = {
    #         'depth': len(self.depth),
    #         'temperature': len(self.temperature),
    #         'hydrostatic_pressure': len(self.hydrostatic_pressure),
    #         'fluid_pressure': len(self.fluid_pressure),
    #         'brine_pressure': len(self.brine_pressure),
    #         'min_horizontal_stress': len(self.min_horizontal_stress),
    #     }
    #     unique_lengths = set(lengths.values())
    #     if len(unique_lengths) != 1:
    #         raise ValueError(f"All arrays must have the same length, but got lengths: {lengths}")
        

    def compute_temperature(self) -> np.ndarray:
        """Compute temperature at a given depth using the geothermal gradient."""

        temp_array = self.ground_temperature + (self.geothermal_gradient * (self.depth - self.ground_elevation) / 1000)

        temp_array[self.depth < self.ground_elevation] = self.ground_temperature

        return temp_array
    
    def compute_hydrostatic_pressure(self) -> np.ndarray:
        pressure_pa = const.atm + self.depth * self.rho_brine * const.g
        pressure_bar = pressure_pa / 1e5
        return pressure_bar
    
    def compute_shmin(self) -> np.ndarray:
        """Compute minimum horizontal stress at a given depth."""
        # Placeholder: linear increase with depth, can be replaced with a more complex model
        
        ground_pressure = np.interp(self.ground_elevation, self.depth, self.hydrostatic_pressure)


        if self.shmin_data:


            return shmin_data_interpolator(shmin_data=self.shmin_data, depth_array=self.depth, ground_elevation=self.ground_elevation, ground_pressure_bar=ground_pressure, surface_pressure_bar=const.atm/1e5)

        else:
            raise ValueError("Either shmin_gradient or shmin_data must be provided.")




    def get_values_at_depth(self, depth_value: float) -> dict:
        """Interpolate all curves at a given depth."""

        def interp(arr):
            f = interp1d(self.depth, arr, bounds_error=False, fill_value="extrapolate")
            return float(f(depth_value))  # convert numpy float to Python float
    
        return {
            "temperature": interp(self.temperature),
            "hydrostatic_pressure": interp(self.hydrostatic_pressure),
            # "fluid_pressure": interp(self.fluid_pressure),
            # "brine_pressure": interp(self.brine_pressure),
            "min_horizontal_stress": interp(self.min_horizontal_stress),
        }
    

# @dataclass
# class Pressure:
#     ground_elevation: float
#     total_depth_msl: float

#     def __post_init__(self):
#         self.pressure_tables: List[PressureTable] = []


In [ ]:
depth_array = np.arange(0, 3001, 1)

pressure_table = PressureTable(
    name="example",
    depth=depth_array,
    ground_elevation=100,
    ground_temperature=4,
    geothermal_gradient=40,
    rho_brine=1030,
    shmin_gradient=0.1695,
    shmin_data=[[100, 5.0], [500, 15.0], [1000, 25.0]],  # optional
)

print(pressure_table.temperature)
print(pressure_table.hydrostatic_pressure)
print(pressure_table.min_horizontal_stress)

In [ ]:
depth_array = np.arange(0, 3001, 1)  # 0 to 3000 m

table = PressureTable(
    name="scenario1",
    depth=depth_array,
    ground_temperature=4.0,
    geothermal_gradient=40.0,
    ground_elevation=350.0,
    shmin_data= data
)

table.get_values_at_depth(800)

In [ ]:
table

In [ ]:
my_pressure = Pressure(ground_elevation=100.0, total_depth_msl=3000.0)
my_pressure.pressure_tables

In [ ]:
@dataclass
class Pressure:
    ground_elevation: float
    total_depth_msl: float

    def __post_init__(self):
        self.pressure_tables: List[PressureTable] = []

In [ ]:
data = [
    [
        333.14687,
        31.07278746476919
    ],
    [
        335.02718,
        31.35629467551348
    ],
    [
        336.90752,
        31.6525861597824
    ],
    [
        338.78786,
        31.936261785300726
    ],
    [
        340.67818,
        32.221735066341544
    ],
    [
        342.55855,
        32.5054383593364
    ],
    [
        344.43886,
        32.8044896353611
    ],
    [
        346.31917,
        33.08889020646915
    ],
    [
        348.19948,
        33.37375125234564
    ],
    [
        350.08986,
        33.66106021851192
    ],
    [
        351.97017,
        33.96028357501336
    ],
    [
        353.85048,
        34.24654009605816
    ],
    [
        355.73079,
        34.53286383019044
    ],
    [
        357.62117,
        34.820218653908036
    ],
    [
        359.50148,
        35.107335246798364
    ],
    [
        361.38182,
        35.4090146741496
    ],
    [
        363.26213,
        35.69623981827058
    ],
    [
        365.14244,
        35.984172927273484
    ],
    [
        367.03283,
        36.27272449481643
    ],
    [
        368.91314,
        36.57616847450244
    ],
    [
        370.79345,
        36.86480650538416
    ],
    [
        372.67376,
        37.15301882070145
    ],
    [
        374.55407,
        37.44225303561567
    ],
    [
        376.44445,
        37.748290213088104
    ],
    [
        378.32476,
        38.03741792741484
    ],
    [
        380.20507,
        38.32754540632921
    ],
    [
        382.08541,
        38.61717637888468
    ],
    [
        383.97576,
        38.925003952008716
    ],
    [
        385.8561,
        39.21517281276
    ],
    [
        387.73641,
        39.50592851412702
    ],
    [
        389.61672,
        39.79726856356536
    ],
    [
        391.49706,
        40.104168785332924
    ],
    [
        393.38741,
        40.3969920782536
    ],
    [
        395.26772,
        40.689347252434196
    ],
    [
        397.14806,
        40.980701938215965
    ],
    [
        399.02837,
        41.28942028388463
    ],
    [
        400.91875,
        41.5837457881875
    ],
    [
        402.79906,
        41.87637469796923
    ],
    [
        404.6794,
        42.1695209679822
    ],
    [
        406.55971,
        42.46237514924766
    ],
    [
        408.44002,
        42.77255366443501
    ],
    [
        410.3304,
        43.0683333579432
    ],
    [
        412.21071,
        43.36193307777382
    ],
    [
        414.09102,
        43.65641081964078
    ],
    [
        415.97133,
        43.967681230658584
    ],
    [
        417.86171,
        44.263823926667314
    ],
    [
        419.74202,
        44.558945656186694
    ],
    [
        421.62233,
        44.835487221132006
    ],
    [
        423.50267,
        45.127259131226666
    ],
    [
        425.38298,
        45.406072234074415
    ],
    [
        427.27336,
        45.703002697421766
    ],
    [
        429.15367,
        45.999274714000734
    ],
    [
        431.03398,
        46.29553479392868
    ],
    [
        432.91429,
        46.6096088042775
    ],
    [
        434.7946,
        46.90674234792721
    ],
    [
        436.68499,
        47.20407098618611
    ],
    [
        438.5653,
        47.500255942075796
    ],
    [
        440.44561,
        47.7959415268708
    ],
    [
        442.32598,
        48.088948053777116
    ],
    [
        444.2163,
        48.4025330090016
    ],
    [
        446.09667,
        48.69538298045271
    ],
    [
        447.97698,
        48.98764954076599
    ],
    [
        449.85729,
        49.28064655638681
    ],
    [
        451.7376,
        49.5632951626752
    ],
    [
        453.62798,
        49.84724354523732
    ],
    [
        455.50829,
        50.14993632072021
    ],
    [
        457.3886,
        50.438615226222595
    ],
    [
        459.26891,
        50.72841610314175
    ],
    [
        461.15929,
        51.018648992821255
    ],
    [
        463.0396,
        51.307979411962805
    ],
    [
        464.91991,
        51.59751410578401
    ],
    [
        466.80022,
        51.90190133303879
    ],
    [
        468.68053,
        52.187289245654576
    ],
    [
        470.57091,
        52.478567158935526
    ],
    [
        472.45122,
        52.77678498268704
    ],
    [
        474.33153,
        53.07617275679952
    ],
    [
        476.2119,
        53.3753412485706
    ],
    [
        478.10222,
        53.676327786832076
    ],
    [
        479.98259,
        53.996205441593254
    ],
    [
        481.8629,
        54.296602135328705
    ],
    [
        483.74321,
        54.59769239268951
    ],
    [
        485.62352,
        54.898999817002554
    ],
    [
        487.5139,
        55.202137361257506
    ],
    [
        489.39421,
        55.50434628607611
    ],
    [
        491.27452,
        55.821699605307245
    ],
    [
        493.15483,
        56.11711189512709
    ],
    [
        495.03514,
        56.41314765437611
    ],
    [
        496.92553,
        56.714369637101136
    ],
    [
        498.80584,
        57.0141135610956
    ],
    [
        500.68615,
        57.31155236168145
    ],
    [
        502.56646,
        57.624894491143316
    ],
    [
        504.45684,
        57.91934273892158
    ],
    [
        506.33715,
        58.212718830659256
    ],
    [
        508.21746,
        58.506171871311
    ],
    [
        510.09777,
        58.80019673303685
    ],
    [
        511.97808,
        59.098811419808634
    ],
    [
        513.86846,
        59.42036386285398
    ],
    [
        515.74877,
        59.721272200308064
    ],
    [
        517.62914,
        60.02431959050606
    ],
    [
        519.50945,
        60.327979813248305
    ],
    [
        521.39989,
        60.63088036447235
    ],
    [
        523.2802,
        60.932179229187604
    ],
    [
        525.16051,
        61.233556867525984
    ],
    [
        527.04082,
        61.55568882751194
    ],
    [
        528.92113,
        61.85728119520395
    ],
    [
        530.81145,
        62.15958721958896
    ],
    [
        532.69176,
        62.459730373717456
    ],
    [
        534.57207,
        62.75991355538293
    ],
    [
        536.45238,
        63.06013123083306
    ],
    [
        538.34282,
        63.36209825565876
    ],
    [
        540.22313,
        63.66290160158783
    ],
    [
        542.10344,
        63.98712587215944
    ],
    [
        543.98375,
        64.29018258178876
    ],
    [
        545.86406,
        64.59219356705893
    ],
    [
        547.75438,
        64.89432814552704
    ],
    [
        549.63469,
        65.19527766580258
    ],
    [
        551.515,
        65.497303151685
    ],
    [
        553.39537,
        65.79987439025386
    ],
    [
        555.28575,
        66.10417590833325
    ],
    [
        557.16606,
        66.40672554096571
    ],
    [
        559.04638,
        66.73284143604918
    ],
    [
        560.92675,
        67.03213604055976
    ],
    [
        562.80706,
        67.32750829999769
    ],
    [
        564.69744,
        67.62455869185072
    ],
    [
        566.57775,
        67.92476733223202
    ],
    [
        568.45806,
        68.2265893250367
    ],
    [
        570.33837,
        68.52835723194657
    ],
    [
        572.21868,
        68.8295041726842
    ],
    [
        574.10906,
        69.15713890229298
    ],
    [
        575.98937,
        69.46048711670014
    ],
    [
        577.86968,
        69.76547083829736
    ],
    [
        579.74999,
        70.07096366510896
    ],
    [
        581.64037,
        70.3804664187406
    ],
    [
        583.52068,
        70.68813106514796
    ],
    [
        585.40099,
        70.99688647347732
    ],
    [
        587.2813,
        71.3061620545257
    ],
    [
        589.16161,
        71.6159578082931
    ],
    [
        591.05199,
        71.9501122517571
    ],
    [
        592.9323,
        72.2581133497038
    ],
    [
        594.81261,
        72.5666161745284
    ],
    [
        596.69292,
        72.87503537047643
    ],
    [
        598.5833,
        73.1863582229682
    ],
    [
        600.46361,
        73.49518946232288
    ],
    [
        602.34398,
        73.80215880417323
    ],
    [
        604.22429,
        74.10960039404773
    ],
    [
        606.1046,
        74.439521354457
    ],
    [
        607.99498,
        74.74326399124472
    ],
    [
        609.87529,
        75.04681085578764
    ],
    [
        611.7556,
        75.3484035807108
    ],
    [
        613.63591,
        75.65042793831569
    ],
    [
        615.52629,
        75.95473036815612
    ],
    [
        617.4066,
        76.25762099151301
    ],
    [
        619.28691,
        76.56094324755162
    ],
    [
        621.16722,
        76.8653065013148
    ],
    [
        623.04753,
        77.17010507692795
    ],
    [
        624.93792,
        77.50295039721887
    ],
    [
        626.81823,
        77.80747028312206
    ],
    [
        628.69854,
        78.11303486580648
    ],
    [
        630.57885,
        78.4184124833208
    ],
    [
        632.45916,
        78.72421804434865
    ],
    [
        634.34954,
        79.03232843687874
    ],
    [
        636.22985,
        79.3371204509949
    ],
    [
        638.11016,
        79.641077368986
    ],
    [
        639.99047,
        79.94544378464953
    ],
    [
        641.88085,
        80.25147870484096
    ],
    [
        643.76116,
        80.55666521238169
    ],
    [
        645.64153,
        80.88950382885218
    ],
    [
        647.52184,
        81.19558865538792
    ],
    [
        649.40215,
        81.50272004310524
    ],
    [
        651.29259,
        81.79811980992054
    ],
    [
        653.1729,
        82.08553554437941
    ],
    [
        655.05321,
        82.37324641229587
    ],
    [
        656.93352,
        82.65738570297121
    ],
    [
        658.82384,
        82.9437045137784
    ],
    [
        660.70415,
        83.22839237260034
    ],
    [
        662.58446,
        83.51270323451536
    ],
    [
        664.46483,
        83.79794281245587
    ],
    [
        666.34514,
        84.10501941761143
    ],
    [
        668.23552,
        84.38426358307198
    ],
    [
        670.11583,
        84.66378022490247
    ],
    [
        671.99614,
        84.96199136347253
    ],
    [
        673.87645,
        85.26054190551886
    ],
    [
        675.76683,
        85.56335855062586
    ],
    [
        677.64714,
        85.86592004602046
    ],
    [
        679.52745,
        86.16950600716082
    ],
    [
        681.40776,
        86.47479042881184
    ],
    [
        683.28814,
        86.7811229632431
    ],
    [
        685.17846,
        87.0884193736719
    ],
    [
        687.05883,
        87.41773717151877
    ],
    [
        688.93914,
        87.71848017196861
    ],
    [
        690.81945,
        88.02226966224826
    ],
    [
        692.69976,
        88.32844823890248
    ],
    [
        694.59014,
        88.63627338131452
    ],
    [
        696.47045,
        88.9411262905152
    ],
    [
        698.35076,
        89.24632967069678
    ],
    [
        700.23107,
        89.54844888846088
    ],
    [
        702.12145,
        89.8335917228988
    ],
    [
        704.00176,
        90.11767918163473
    ],
    [
        705.88207,
        90.39438188455115
    ],
    [
        707.76238,
        90.66294464547761
    ],
    [
        709.64269,
        90.96019751176739
    ],
    [
        711.53307,
        91.25066458057742
    ],
    [
        713.41338,
        91.54429460612712
    ],
    [
        715.29369,
        91.840306428623
    ],
    [
        717.174,
        92.14997341242601
    ],
    [
        719.06438,
        92.46129310209528
    ],
    [
        720.94469,
        92.77167674276397
    ],
    [
        722.825,
        93.08241823275002
    ],
    [
        724.70531,
        93.39351757205337
    ],
    [
        726.58568,
        93.70498249864514
    ],
    [
        728.476,
        94.018089446316
    ],
    [
        730.35637,
        94.32955456159328
    ],
    [
        732.23668,
        94.63849279660693
    ],
    [
        734.11699,
        94.94849060386399
    ],
    [
        736.00737,
        95.29118184498067
    ],
    [
        737.88774,
        95.60195384705874
    ],
    [
        739.76799,
        95.91377910678138
    ],
    [
        741.64836,
        96.22596670678044
    ],
    [
        743.52861,
        96.53921492062914
    ],
    [
        745.41899,
        96.85339855415712
    ],
    [
        747.2993,
        97.165155286782
    ],
    [
        749.17961,
        97.47725142288313
    ],
    [
        751.05992,
        97.78526622377137
    ],
    [
        752.95031,
        98.09047965729978
    ],
    [
        754.83068,
        98.39468366619624
    ],
    [
        756.71093,
        98.70213649984147
    ],
    [
        758.5913,
        99.00991487590382
    ],
    [
        760.47155,
        99.31873350251205
    ],
    [
        762.36193,
        99.66284400380914
    ],
    [
        764.2423,
        99.9738887582124
    ],
    [
        766.12261,
        100.28599819278877
    ],
    [
        768.00292,
        100.599185685033
    ],
    [
        769.88329,
        100.91270937776888
    ],
    [
        771.77361,
        101.22710937533985
    ],
    [
        773.65398,
        101.53977087594585
    ],
    [
        775.53423,
        101.85349838362552
    ],
    [
        777.4146,
        102.16680365438638
    ],
    [
        779.30498,
        102.48251074655977
    ],
    [
        781.18523,
        102.7972112039082
    ],
    [
        783.0656,
        103.11225209902082
    ],
    [
        784.94591,
        103.42760974545637
    ],
    [
        786.82622,
        103.74174828565164
    ],
    [
        788.7166,
        104.09002935823798
    ],
    [
        790.59691,
        104.40410518189665
    ],
    [
        792.47722,
        104.71693974454837
    ],
    [
        794.35753,
        105.02930124342541
    ],
    [
        796.24791,
        105.34485974968946
    ],
    [
        798.12822,
        105.65939669894138
    ],
    [
        800.00853,
        105.97424353832375
    ],
    [
        801.88884,
        106.28782696193258
    ],
    [
        803.76915,
        106.60092439979925
    ],
    [
        805.65953,
        106.91486665525575
    ],
    [
        807.53984,
        107.22301531788096
    ],
    [
        809.42015,
        107.5322310201216
    ],
    [
        811.30053,
        107.84412037215485
    ],
    [
        813.19085,
        108.16001277025457
    ],
    [
        815.07122,
        108.47488137226851
    ],
    [
        816.95153,
        108.78843795759099
    ],
    [
        818.83184,
        109.13522022263955
    ],
    [
        820.71215,
        109.447825087251
    ],
    [
        822.60253,
        109.76044373870896
    ],
    [
        824.48284,
        110.07199617138362
    ],
    [
        826.36315,
        110.38058264267416
    ],
    [
        828.24346,
        110.68943104490833
    ],
    [
        830.12377,
        110.99854137808612
    ],
    [
        832.01415,
        111.3125256594747
    ],
    [
        833.89446,
        111.62544013612278
    ],
    [
        835.77477,
        111.937811405338
    ],
    [
        837.65508,
        112.2504556730015
    ],
    [
        839.54546,
        112.56472310633552
    ],
    [
        841.42577,
        112.87791410191714
    ],
    [
        843.30608,
        113.19137809594704
    ],
    [
        845.18645,
        113.50512314618267
    ],
    [
        847.0667,
        113.81829410691898
    ],
    [
        848.95708,
        114.13392906416112
    ],
    [
        850.83739,
        114.44848393842163
    ],
    [
        852.71782,
        114.79846164099229
    ],
    [
        854.59807,
        115.11363191144437
    ],
    [
        856.48845,
        115.42960019421044
    ],
    [
        858.36876,
        115.74448136583625
    ],
    [
        860.24907,
        116.05878794240442
    ],
    [
        862.12938,
        116.37420588800622
    ],
    [
        864.00975,
        116.68905365269727
    ],
    [
        865.90013,
        117.00467020803727
    ],
    [
        867.78038,
        117.31918057627017
    ],
    [
        869.66075,
        117.63396909879303
    ],
    [
        871.541,
        117.94900332055501
    ],
    [
        873.43138,
        118.26567110241828
    ],
    [
        875.31169,
        118.58123800330534
    ],
    [
        877.19207,
        118.89363422144987
    ],
    [
        879.07238,
        119.20626813074419
    ],
    [
        880.95269,
        119.5113712830092
    ],
    [
        882.84307,
        119.81112653897611
    ],
    [
        884.72332,
        120.1452764530356
    ],
    [
        886.60369,
        120.43976949926775
    ],
    [
        888.48394,
        120.73266905130252
    ],
    [
        890.36431,
        121.02923733346218
    ],
    [
        892.25469,
        121.34046968332804
    ],
    [
        894.135,
        121.64968477727999
    ],
    [
        896.01537,
        121.95913307725952
    ],
    [
        897.89568,
        122.26879825332479
    ],
    [
        899.78606,
        122.5800603294331
    ],
    [
        901.66637,
        122.88663804778419
    ],
    [
        903.54668,
        123.19253966943073
    ],
    [
        905.42699,
        123.49864788449763
    ],
    [
        907.3073,
        123.80229248760091
    ],
    [
        909.19768,
        124.10839911695977
    ],
    [
        911.07799,
        124.41690597758897
    ],
    [
        912.9583,
        124.72741803415948
    ],
    [
        914.83861,
        125.03815144082326
    ],
    [
        916.72899,
        125.34688589144139
    ],
    [
        918.6093,
        125.6544505897821
    ],
    [
        920.48961,
        125.96041588092832
    ],
    [
        922.36992,
        126.2982499583616
    ],
    [
        924.25023,
        126.59833141379
    ],
    [
        926.14061,
        126.90087459741675
    ],
    [
        928.02092,
        127.20130514353943
    ],
    [
        929.90123,
        127.50099684746186
    ],
    [
        931.78154,
        127.80268640850385
    ],
    [
        933.67192,
        128.1068506520748
    ],
    [
        935.55223,
        128.40797891528857
    ],
    [
        937.43261,
        128.7092901803932
    ],
    [
        939.31292,
        129.00984376267257
    ],
    [
        941.19323,
        129.3096437361315
    ],
    [
        943.08361,
        129.61006850941254
    ],
    [
        944.96392,
        129.9101991079673
    ],
    [
        946.84423,
        130.20956686490229
    ],
    [
        948.72454,
        130.51002764401272
    ],
    [
        950.61492,
        130.81110759313344
    ],
    [
        952.49523,
        131.11003020169844
    ],
    [
        954.37554,
        131.4081752020922
    ],
    [
        956.25585,
        131.71022749550656
    ],
    [
        958.13616,
        132.01432935380498
    ],
    [
        960.02654,
        132.31999623543427
    ],
    [
        961.90685,
        132.62162183619841
    ],
    [
        963.78716,
        132.9621779328948
    ],
    [
        965.66753,
        133.26706142641856
    ],
    [
        967.54778,
        133.5759020968914
    ],
    [
        969.43816,
        133.88728464217368
    ],
    [
        971.31853,
        134.20033936818328
    ],
    [
        973.19878,
        134.51549352405246
    ],
    [
        975.07915,
        134.8308782380733
    ],
    [
        976.96953,
        135.15361209419967
    ],
    [
        978.84978,
        135.47518130501695
    ],
    [
        980.73015,
        135.79604113461392
    ],
    [
        982.6104,
        136.11711676610162
    ],
    [
        984.49077,
        136.439407223756
    ],
    [
        986.38115,
        136.76235405828842
    ],
    [
        988.2614,
        137.0831599138932
    ],
    [
        990.14177,
        137.40421114330022
    ],
    [
        992.02202,
        137.7176890596527
    ],
    [
        993.35955,
        137.94137691790814
    ],
    [
        994.15374,
        138.07409200624943
    ],
    [
        994.93804,
        138.20546912696676
    ],
    [
        995.73247,
        138.32852087734886
    ],
    [
        996.5269,
        138.45159289046248
    ],
    [
        997.32122,
        138.57466988215117
    ],
    [
        998.10552,
        138.69539561674802
    ],
    [
        998.89995,
        138.85968418386975
    ],
    [
        999.69414,
        138.9818550040508
    ],
    [
        1000.48857,
        139.1030964120298
    ],
    [
        1001.27288,
        139.22196557159666
    ],
    [
        1002.06731,
        139.34127432574917
    ],
    [
        1002.86162,
        139.46156422767953
    ],
    [
        1003.65593,
        139.5818697139721
    ],
    [
        1004.45037,
        139.7022088654477
    ],
    [
        1005.23455,
        139.82113650917384
    ],
    [
        1006.02898,
        139.94249225722524
    ],
    [
        1006.82341,
        140.0638651506649
    ],
    [
        1007.6176,
        140.185221799392
    ],
    [
        1008.40203,
        140.30523761890834
    ],
    [
        1009.19634,
        140.4266451403727
    ],
    [
        1009.99065,
        140.54806980463545
    ],
    [
        1010.78508,
        140.6695283119267
    ],
    [
        1011.56938,
        140.7886017279757
    ],
    [
        1012.36382,
        140.9100950292867
    ],
    [
        1013.15801,
        141.03057676819998
    ],
    [
        1013.95244,
        141.15110749923423
    ],
    [
        1014.74688,
        141.27265067586052
    ],
    [
        1015.53105,
        141.39377733797642
    ],
    [
        1016.32549,
        141.51635248963862
    ],
    [
        1017.1198,
        141.6399260369814
    ],
    [
        1017.91411,
        141.76252127025316
    ],
    [
        1018.69841,
        141.88374101759518
    ],
    [
        1019.49285,
        142.0493969102472
    ],
    [
        1020.28728,
        142.17209835871392
    ],
    [
        1021.08147,
        142.295786746498
    ],
    [
        1021.86589,
        142.41713128900824
    ],
    [
        1022.66021,
        142.53987417636682
    ],
    [
        1023.45452,
        142.66263437126315
    ],
    [
        1024.24895,
        142.78643478421466
    ],
    [
        1025.04326,
        142.90923316221108
    ],
    [
        1025.82756,
        143.03166088868318
    ],
    [
        1026.62188,
        143.15449872495202
    ],
    [
        1027.41631,
        143.27837849803717
    ],
    [
        1028.21074,
        143.40227853385383
    ],
    [
        1028.99492,
        143.52375971283013
    ],
    [
        1029.78936,
        143.64770075897906
    ],
    [
        1030.58367,
        143.7716439324559
    ],
    [
        1031.37798,
        143.89459558380526
    ],
    [
        1032.16228,
        144.0161692549931
    ],
    [
        1032.95671,
        144.14018826635242
    ],
    [
        1033.75103,
        144.26319807973007
    ],
    [
        1034.54546,
        144.38624194671766
    ],
    [
        1035.33977,
        144.50827210024923
    ],
    [
        1036.12419,
        144.62792244836274
    ],
    [
        1036.91838,
        144.74793481613247
    ],
    [
        1039.0739,
        145.07533582247163
    ],
    [
        1041.9644,
        145.5075276596928
    ],
    [
        1044.86467,
        145.93919288754006
    ],
    [
        1047.76519,
        146.35973455437625
    ],
    [
        1050.6657,
        146.823649714665
    ],
    [
        1053.5562,
        147.24411477500522
    ],
    [
        1056.45659,
        147.66190741144962
    ],
    [
        1059.3571,
        148.07354973270841
    ],
    [
        1062.25762,
        148.48835382077348
    ],
    [
        1065.15813,
        148.90216280815056
    ],
    [
        1068.04851,
        149.3198399124114
    ],
    [
        1070.94902,
        149.73795650405413
    ],
    [
        1073.84941,
        150.156124605731
    ],
    [
        1076.74993,
        150.57543546527282
    ],
    [
        1079.64031,
        150.99446130004216
    ],
    [
        1082.54082,
        151.4160453834036
    ],
    [
        1085.44133,
        151.83771482877438
    ],
    [
        1088.34185,
        152.25626804509395
    ],
    [
        1091.24224,
        152.67165983714307
    ],
    [
        1094.13274,
        153.07927901838488
    ],
    [
        1097.03325,
        153.49046811328802
    ],
    [
        1099.93352,
        153.90381015118874
    ],
    [
        1102.83403,
        154.31722560408835
    ],
    [
        1105.73455,
        154.72851284075264
    ],
    [
        1108.62505,
        155.13516367236227
    ],
    [
        1111.52544,
        155.54539063323935
    ],
    [
        1114.42595,
        155.95784365320225
    ],
    [
        1117.32646,
        156.37142691522612
    ],
    [
        1120.21697,
        156.7847493628119
    ],
    [
        1123.11736,
        157.1983982768383
    ],
    [
        1126.01787,
        157.61320844511192
    ],
    [
        1128.91826,
        158.0302622795967
    ],
    [
        1131.81877,
        158.44961044739262
    ],
    [
        1134.70916,
        158.86983632989717
    ],
    [
        1137.60967,
        159.33173446185418
    ],
    [
        1140.51018,
        159.7469258334058
    ],
    [
        1143.41069,
        160.16664947491
    ],
    [
        1146.30107,
        160.585022296925
    ],
    [
        1149.20159,
        161.00488379940884
    ],
    [
        1152.1021,
        161.42820282708303
    ],
    [
        1155.00249,
        161.85385651593845
    ],
    [
        1157.90288,
        162.28983006892946
    ],
    [
        1160.79338,
        162.72456515765626
    ],
    [
        1163.6939,
        163.1597110112916
    ],
    [
        1166.59429,
        163.59040318915305
    ],
    [
        1169.4948,
        164.01778987564438
    ],
    [
        1172.3853,
        164.45192572014838
    ],
    [
        1175.28582,
        164.88760910712247
    ],
    [
        1178.18621,
        165.3211049216224
    ],
    [
        1181.08672,
        165.75243116788226
    ],
    [
        1183.98711,
        166.18037560374827
    ],
    [
        1186.87773,
        166.6105439946905
    ],
    [
        1189.778,
        167.04335349572398
    ],
    [
        1192.67852,
        167.4716432185843
    ],
    [
        1195.57903,
        167.9012068348967
    ],
    [
        1198.47954,
        168.3308785764208
    ],
    [
        1201.36992,
        168.75569982674878
    ],
    [
        1204.27043,
        169.1784907579315
    ],
    [
        1207.17095,
        169.60017283976927
    ],
    [
        1210.07134,
        170.02190494571056
    ],
    [
        1212.96184,
        170.46611393206533
    ],
    [
        1215.86223,
        170.91547290029322
    ],
    [
        1218.76274,
        171.369830335984
    ],
    [
        1221.66313,
        171.8255912866777
    ],
    [
        1224.56365,
        172.28039685249783
    ],
    [
        1227.45415,
        172.72919776140404
    ],
    [
        1230.35466,
        173.17960612692372
    ],
    [
        1233.25505,
        173.63745571486814
    ],
    [
        1236.15557,
        174.1477016751112
    ],
    [
        1239.04607,
        174.60717756434553
    ],
    [
        1241.94658,
        175.06952670105616
    ],
    [
        1244.84685,
        175.534534780839
    ],
    [
        1247.74736,
        176.0010625089259
    ],
    [
        1250.64788,
        176.46663223008326
    ],
    [
        1253.53838,
        176.9285900107741
    ],
    [
        1256.43877,
        177.39342622347718
    ],
    [
        1259.33928,
        177.85730004757485
    ],
    [
        1262.23979,
        178.32885381030385
    ],
    [
        1265.1303,
        178.80176101125244
    ],
    [
        1268.03069,
        179.27014027445233
    ],
    [
        1270.9312,
        179.74129753198085
    ],
    [
        1273.83159,
        180.21271666198524
    ],
    [
        1276.73198,
        180.68190968153885
    ],
    [
        1279.62249,
        181.14620556481776
    ],
    [
        1282.523,
        181.60839256873498
    ],
    [
        1285.42351,
        182.06955189501437
    ],
    [
        1288.3239,
        182.5321856973534
    ],
    [
        1291.2144,
        182.991117818376
    ],
    [
        1294.11492,
        183.44915248061304
    ],
    [
        1297.01543,
        183.90739628527135
    ],
    [
        1299.91582,
        184.36583363006594
    ],
    [
        1302.81621,
        184.8206473376661
    ],
    [
        1305.70671,
        185.27425117537643
    ],
    [
        1308.60723,
        185.72561873124528
    ],
    [
        1311.50762,
        186.17971742613577
    ],
    [
        1314.40813,
        186.63402094858776
    ],
    [
        1317.30864,
        187.09238910678766
    ],
    [
        1320.19915,
        187.54954151827994
    ],
    [
        1323.09954,
        188.01479181258398
    ],
    [
        1326.00005,
        188.48029247709977
    ],
    [
        1328.90044,
        188.94600940257502
    ],
    [
        1331.79106,
        189.41187661504307
    ],
    [
        1431.65251,
        207.54276312161025
    ],
    [
        1522.19463,
        223.59522247751204
    ],
    [
        1576.51592,
        234.02577962312643
    ],
    [
        1615.14885,
        242.2688436239306
    ],
    [
        1740.35601,
        270.5387611802984
    ],
    [
        1800.28008,
        284.77602265538303
    ],
    [
        1832.29192,
        293.4059898664166
    ],
    [
        1860.16912,
        300.80430246372487
    ],
    [
        1879.325,
        306.263636706825
    ],
    [
        1911.79717,
        314.4080486508504
    ],
    [
        1921.08965,
        316.5280211235474
    ],
    [
        1939.67437,
        320.52822000103953
    ],
    [
        1975.81243,
        328.12231274750536
    ],
    [
        1983.29753,
        329.85370939008146
    ],
    [
        2040.34746,
        342.6746460958286
    ],
    [
        2065.13018,
        348.5730732014483
    ],
    [
        2086.80938,
        353.7758613876072
    ],
    [
        2109.63115,
        359.68126124199557
    ],
    [
        2113.75713,
        360.80565954822004
    ],
    [
        2119.5374,
        362.4535236038293
    ],
    [
        2134.82305,
        366.8015092749093
    ],
    [
        2147.32085,
        370.3897199794455
    ],
    [
        2250.46611,
        401.81155104063566
    ],
    [
        2321.50005,
        421.31743657425005
    ],
    [
        2321.5999,
        373.50827831160007
    ],
    [
        2373.50005,
        381.8581820442001
    ],
    [
        2373.5999,
        381.8742463116001
    ],
    [
        2406.50005,
        387.16735404420007
    ],
    [
        2406.5999,
        389.5442928135
    ],
    [
        2441.50005,
        392.7982940442001
    ],
    [
        2441.5999,
        395.20956781350003
    ],
    [
        2478.50005,
        401.18241059325004
    ],
    [
        2478.5999,
        401.19857281350005
    ],
    [
        2520.5999,
        410.4696113154
    ],
    [
        2552.50005,
        413.16042059325
    ],
    [
        2552.5999,
        415.6806833154
    ],
    [
        2560.50005,
        411.9434900442001
    ],
    [
        2589.5999,
        419.16558781350005
    ],
    [
        2594.5999,
        437.7920303268001
    ],
    [
        2632.5999,
        444.2038463268001
    ]
]

In [ ]:
import warnings
from scipy.interpolate import RectBivariateSpline, interp1d

depth_array = np.linspace(0,2700,2701)

ground_elevation = 500
ground_pressure_bar = (const.atm + ground_elevation * const.g * 1030) / 1e5
surface_pressure_bar = const.atm/1e5

def shmin_data_interpolator(shmin_data:List[List[float]], depth_array:np.ndarray, ground_elevation:float, ground_pressure_bar:float, surface_pressure_bar:float) -> np.ndarray:

    shmin_data = np.array(shmin_data)
    depth_values, shmin_values = shmin_data.T

    # Remove duplicates while keeping the first occurrence
    unique_indices = np.unique(depth_values, return_index=True)[1]
    depth_values = depth_values[np.sort(unique_indices)]
    shmin_values = shmin_values[np.sort(unique_indices)]


    if min(depth_values) > ground_elevation:
        warnings.warn(
            f"No Shmin data between seafloor depth ({ground_elevation}) and minimum provided depth ({min(depth_values)}). "
            "Extrapolating using hydrostatic pressure at seafloor."
        )

    else:
        filtered_depth_values = depth_values[depth_values >= ground_elevation]
        filtered_shmin_values = shmin_values[depth_values >= ground_elevation]

        depth_values = filtered_depth_values
        shmin_values = filtered_shmin_values

    # Insert the seafloor depth and pressure at mudline into the arrays
    depth_values = np.insert(depth_values, 0, ground_elevation)
    shmin_values = np.insert(shmin_values, 0, ground_pressure_bar)


    # Insert values af MSL
    depth_values = np.insert(depth_values, 0, 0)
    shmin_values = np.insert(shmin_values, 0, surface_pressure_bar)


    shmin_interpolator = interp1d(depth_values, shmin_values, bounds_error=False, fill_value="extrapolate")

    shmin_curve = shmin_interpolator(depth_array)
    return shmin_curve

shmin_curve = shmin_data_interpolator(shmin_data, depth_array, ground_elevation, ground_pressure_bar, surface_pressure_bar)


In [ ]:
import matplotlib.pyplot as plt
# plt.plot(np.array(data)[:,0], np.array(data)[:,1], c='r', lw=4, zorder = 10)

plt.plot(depth_values, shmin_values, 'o', label="Shmin Data Points")
plt.plot(depth_array, shmin_curve
         , label="Interpolated Shmin")

In [ ]:
data